# Microsoft Fabric Capacity Scaling (Best Practice)

This notebook scales a Microsoft Fabric capacity to a target SKU using Azure Resource Manager.

## Best-Practice Approach
- **No Service Principal or client secrets** — uses the Fabric **workspace managed identity**
- **No Key Vault dependency** — no secrets to store, rotate, or expire
- **No user-identity dependency** — runs the same regardless of who triggers it
- **Retry logic** for transient ARM failures

## Prerequisites
1. **Enable workspace identity** on your Fabric workspace (Workspace Settings → Identity)
2. Grant the workspace identity **Contributor** role on the Fabric capacity resource in Azure IAM
3. Run this notebook in the Fabric workspace where the identity is enabled

## Step 1: Configuration
Update these values to match your environment.

In [ ]:
# Configuration
SUBSCRIPTION_ID = "<your-azure-subscription-id>"
RESOURCE_GROUP  = "<your-resource-group-name>"
CAPACITY_NAME   = "<your-fabric-capacity-name>"
TARGET_SKU      = "F128"        # Target SKU: F2, F4, F8, F16, F32, F64, F128, F256, etc.
API_VERSION     = "2023-11-01"  # ARM API version for Microsoft.Fabric/capacities

## Step 2: Authenticate Using Workspace Managed Identity
The workspace identity is tied to the Fabric workspace, not a person.
No secrets, no expiration, no user dependency.

In [ ]:
from notebookutils import mssparkutils

# Get an ARM token using the workspace managed identity.
# This does NOT depend on who runs the notebook.
arm_token = mssparkutils.credentials.getToken("https://management.azure.com")

headers = {
    "Authorization": f"Bearer {arm_token}",
    "Content-Type":  "application/json"
}

print("Authenticated using workspace managed identity.")

## Step 3: Verify Current SKU
Read the capacity directly by resource group and name (no subscription-wide scan needed).

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# HTTP session with automatic retries on transient errors
session = requests.Session()
retries = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "PATCH"]
)
session.mount("https://", HTTPAdapter(max_retries=retries))

# Build the resource URL directly (no subscription-wide scan)
capacity_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.Fabric/capacities/{CAPACITY_NAME}"
    f"?api-version={API_VERSION}"
)

response = session.get(capacity_url, headers=headers)
response.raise_for_status()

capacity_data = response.json()
current_sku = capacity_data["sku"]["name"]

print(f"Capacity:    {capacity_data['name']}")
print(f"Resource ID: {capacity_data['id']}")
print(f"Location:    {capacity_data['location']}")
print(f"Current SKU: {current_sku}")
print(f"Target SKU:  {TARGET_SKU}")

## Step 4: Scale the Capacity
Sends a PATCH request to update the SKU, then polls until the change is confirmed.

In [ ]:
import json
import time

POLL_INTERVAL_SECONDS = 15
MAX_POLL_ATTEMPTS     = 40   # 40 x 15s = 10 minutes max wait

if current_sku == TARGET_SKU:
    print(f"No action needed - already on {TARGET_SKU}.")
else:
    print(f"Scaling {CAPACITY_NAME}: {current_sku} -> {TARGET_SKU}")

    payload = json.dumps({"sku": {"name": TARGET_SKU, "tier": "Fabric"}})
    patch_response = session.patch(capacity_url, headers=headers, data=payload)
    patch_response.raise_for_status()

    print(f"Scale request accepted (HTTP {patch_response.status_code}). Polling for completion...")

    for attempt in range(1, MAX_POLL_ATTEMPTS + 1):
        time.sleep(POLL_INTERVAL_SECONDS)
        check = session.get(capacity_url, headers=headers).json()
        provisioning_state = check.get("properties", {}).get("provisioningState", "Unknown")
        check_sku = check["sku"]["name"]

        print(f"  Poll {attempt}/{MAX_POLL_ATTEMPTS}: SKU={check_sku}, State={provisioning_state}")

        if check_sku == TARGET_SKU and provisioning_state == "Succeeded":
            print(f"\nScale complete: {CAPACITY_NAME} is now {TARGET_SKU}.")
            break
    else:
        raise TimeoutError(
            f"Scale did not complete within {MAX_POLL_ATTEMPTS * POLL_INTERVAL_SECONDS // 60} minutes. "
            f"Last state: SKU={check_sku}, ProvisioningState={provisioning_state}. "
            f"Check the Azure Portal for the capacity status."
        )